In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

data_dir = "C:/Users/HP/Documents/Cassava_Dataset/cassava_split"  
img_size = (224, 224)
batch_size = 32

# Train dataset
train_ds = image_dataset_from_directory(
    "C:/Users/HP/Documents/Cassava_Dataset/cassava_split/train",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    color_mode="rgb" 
)

# Validation dataset
val_ds = image_dataset_from_directory(
    "C:/Users/HP/Documents/Cassava_Dataset/cassava_split/val",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    color_mode="rgb" 
)

# Test dataset 
test_ds = image_dataset_from_directory(
    "C:/Users/HP/Documents/Cassava_Dataset/cassava_split/test",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    color_mode="rgb" 
)

normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

Found 14977 files belonging to 5 classes.
Found 4279 files belonging to 5 classes.
Found 2141 files belonging to 5 classes.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# Load base model
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights=None,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

# Get number of classes correctly
for images, labels in train_ds.take(1):
    num_classes = labels.shape[1] if len(labels.shape) > 1 else tf.reduce_max(labels).numpy() + 1

# Build model
model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",/
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

Epoch 1/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 581s 1s/step - accuracy: 0.6135 - loss: 1.4608 - val_accuracy: 0.6151 - val_loss: 1.3412
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 529s 1s/step - accuracy: 0.6149 - loss: 1.2770 - val_accuracy: 0.6151 - val_loss: 1.2302
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 540s 1s/step - accuracy: 0.6149 - loss: 1.2103 - val_accuracy: 0.6151 - val_loss: 1.1962
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 527s 1s/step - accuracy: 0.6149 - loss: 1.1910 - val_accuracy: 0.6151 - val_loss: 1.1869
Epoch 5/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 525s 1s/step - accuracy: 0.6149 - loss: 1.1858 - val_accuracy: 0.6151 - val_loss: 1.1842
Epoch 6/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 617s 1s/step - accuracy: 0.6149 - loss: 1.1841 - val_accuracy: 0.6151 - val_loss: 1.1833
Epoch 7/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 577s 1s/step - accuracy: 0.6149 - loss: 1.1836 - val_accuracy: 0.6151 - val_loss: 1.1830
Epoch 8/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 569s 1s/step - accuracy: 0.6149 - loss: 1.1834 - val_accu

In [3]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

67/67 ━━━━━━━━━━━━━━━━━━━━ 66s 905ms/step - accuracy: 0.6147 - loss: 1.1837
Test accuracy: 0.6146660447120667


In [4]:
model.save("cassava_model.h5")

In [6]:
from tensorflow.keras.models import load_model

model = load_model("cassava_model.h5")

# Re-compile if you plan to train or evaluate
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [7]:
model.evaluate(test_ds)

67/67 ━━━━━━━━━━━━━━━━━━━━ 504s 3s/step - accuracy: 0.6147 - loss: 1.1837  


[1.1837079524993896, 0.6146660447120667]

In [9]:
from tensorflow.keras.models import load_model
from PIL import Image
import numpy as np

# Load model
model = load_model("cassava_model.h5")

class_names = [
    "Cassava___bacterial_blight",
    "Cassava___brown_streak_disease",
    "Cassava___green_mottle",
    "Cassava___mosaic_disease",
    "Cassava___healthy"
]

def predict_disease(img_path):
    img = Image.open(img_path).resize((224,224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)
    class_idx = np.argmax(preds)
    confidence = np.max(preds)
    
    return class_names[class_idx], confidence

In [11]:
print(predict_disease(r"C:\Users\HP\Documents\Cassava_Dataset\cassava_split\val\Cassava___mosaic_disease\30339454.jpg"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 24s 24s/step
('Cassava___healthy', np.float32(0.6145137))


## Streamlit UI

In [1]:
pip install googletrans==4.0.0-rc1

Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -

  DEPRECATION: Building 'googletrans' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'googletrans'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyterlab 4.3.4 requires httpx>=0.25.0, but you have httpx 0.13.3 which is incompatible.


In [2]:
pip install streamlit tensorflow pillow gTTS

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install deep-translator

Note: you may need to restart the kernel to use updated packages.Collecting deep-translator



In [5]:
# app.py
import streamlit as st
import numpy as np
from PIL import Image
import io, os, hashlib, datetime, csv, json
from tensorflow.keras.models import load_model
from gtts import gTTS
from deep_translator import GoogleTranslator
import pandas as pd

# -------- CONFIG --------
MODEL_PATH = "cassava_model.h5"     # path to your trained model
AUDIO_CACHE_DIR = "audio_cache"
QUEUE_DIR = "queued_images"
QUEUE_INDEX = os.path.join(QUEUE_DIR, "queue_metadata.csv")
TRANSLATIONS_CACHE = "translations_cache.json"
CONFIDENCE_THRESHOLD = 0.60

# Ensure folders exist
os.makedirs(AUDIO_CACHE_DIR, exist_ok=True)
os.makedirs(QUEUE_DIR, exist_ok=True)
if not os.path.exists(QUEUE_INDEX):
    with open(QUEUE_INDEX, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp","filename","predicted_class","confidence","notes"])

if not os.path.exists(TRANSLATIONS_CACHE):
    with open(TRANSLATIONS_CACHE, "w", encoding="utf-8") as jf:
        json.dump({}, jf)

# -------- CLASS NAMES --------
CLASS_NAMES = [
    "Cassava___bacterial_blight",
    "Cassava___brown_streak_disease",
    "Cassava___green_mottle",
    "Cassava___mosaic_disease",
    "Cassava___healthy"
]

# -------- ENGLISH ADVICE --------
ADVICE = {
    "Cassava___bacterial_blight": {
        "Small Farmland": (
            "Short-term (smallholder): Remove and destroy infected plants immediately. "
            "Burn or deeply bury infected debris and avoid replanting from symptomatic material. "
            "Use only clean cuttings, disinfect tools after use, and avoid overhead watering."
        ),
        "Big Farmland": (
            "Long-term (large farm): Adopt resistant varieties where available, implement certified clean seed systems, "
            "practice crop rotation, and institutionalize tool and equipment disinfection protocols. "
            "Train staff on early detection and coordinate with extension services for area-wide management."
        )
    },
    "Cassava___brown_streak_disease": {
        "Small Farmland": (
            "Short-term (smallholder): Uproot and destroy plants showing brown streak symptoms. "
            "Do not reuse infected cuttings. Remove and burn infected stems after harvest."
        ),
        "Big Farmland": (
            "Long-term (large farm): Source and plant certified virus-free or tolerant varieties. "
            "Invest in clean-seed propagation (tissue culture/greenhouse) and community phytosanitation campaigns."
        )
    },
    "Cassava___green_mottle": {
        "Small Farmland": (
            "Short-term (smallholder): Isolate and destroy symptomatic plants, avoid sharing unclean tools, "
            "and replant only with verified healthy cuttings."
        ),
        "Big Farmland": (
            "Long-term (large farm): Implement systematic surveillance, strict tool disinfection protocols, "
            "and integrate pest management (monitoring traps, biological controls)."
        )
    },
    "Cassava___mosaic_disease": {
        "Small Farmland": (
            "Short-term (smallholder): Use virus-free cuttings, remove mosaic-infected plants promptly, "
            "and reduce whitefly populations by intercropping or cultural controls."
        ),
        "Big Farmland": (
            "Long-term (large farm): Deploy CMD-resistant cultivars, establish a clean seed program, "
            "integrate vector control strategies, and collaborate with research/extension services."
        )
    },
    "Cassava___healthy": {
        "Small Farmland": (
            "Short-term (smallholder): Maintain good practices — use clean cuttings, weed regularly, "
            "monitor crops weekly, and maintain soil fertility."
        ),
        "Big Farmland": (
            "Long-term (large farm): Implement integrated pest management, maintain strict sanitation, "
            "train workers on early detection, and use certified planting materials."
        )
    }
}

LANG_CODE = {"English": "en", "Hausa": "ha", "Yoruba": "yo"}

# -------- UTILITIES --------
@st.cache_resource(show_spinner=False)
def load_prediction_model(path):
    try:
        m = load_model(path)
        return m
    except Exception as e:
        st.error(f"Failed to load model: {e}")
        return None

def preprocess_pil_image(pil_img, target_size=(224,224)):
    if pil_img.mode != "RGB":
        pil_img = pil_img.convert("RGB")
    pil_img = pil_img.resize(target_size)
    arr = np.array(pil_img).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)
    return arr

def predict_from_pil(pil_img, model):
    x = preprocess_pil_image(pil_img)
    preds = model.predict(x)
    class_idx = int(np.argmax(preds, axis=1)[0])
    confidence = float(np.max(preds))
    predicted = CLASS_NAMES[class_idx]
    return predicted, confidence, preds[0]

def translate_text_cached(english_text, target_code, cache_file=TRANSLATIONS_CACHE):
    if target_code == "en":  
        return english_text

    try:
        with open(cache_file, "r", encoding="utf-8") as jf:
            cache = json.load(jf)
    except Exception:
        cache = {}

    key = hashlib.md5((english_text + target_code).encode("utf-8")).hexdigest()
    if key in cache:
        return cache[key]

    try:
        translated = GoogleTranslator(source="en", target=target_code).translate(english_text)
        cache[key] = translated
        with open(cache_file, "w", encoding="utf-8") as jf:
            json.dump(cache, jf, ensure_ascii=False, indent=2)
        return translated
    except Exception as e:
        st.warning(f"Translation failed ({target_code}): using English fallback. ({e})")
        return english_text

def text_to_speech_cached(text, lang_code, cache_dir=AUDIO_CACHE_DIR):
    key = hashlib.md5((text + lang_code).encode("utf-8")).hexdigest()
    fname = f"{key}_{lang_code}.mp3"
    fpath = os.path.join(cache_dir, fname)
    if os.path.exists(fpath):
        return fpath
    try:
        tts = gTTS(text=text, lang=lang_code)
        tts.save(fpath)
        return fpath
    except Exception as e:
        st.warning(f"TTS generation failed: {e}")
        return None

def queue_for_review(pil_img, predicted, confidence, notes=""):
    ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%S")
    fname = f"{predicted}_{int(confidence*100)}_{ts}.jpg"
    path = os.path.join(QUEUE_DIR, fname)
    pil_img.save(path, format="JPEG")
    with open(QUEUE_INDEX, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([ts, fname, predicted, f"{confidence:.4f}", notes])
    return path

# -------- STREAMLIT UI --------
st.set_page_config(page_title="AgriCare - Crop Diagnostic (Demo)", layout="centered")
st.title("AgriCare — Crop Disease Detection & Advice")

farmland_size = st.selectbox("Select Farmland Size:", ["Small Farmland", "Big Farmland"])
language = st.selectbox("Select Language:", ["English", "Hausa", "Yoruba"])
uploaded_file = st.file_uploader("Upload Cassava Leaf Image", type=["jpg","jpeg","png"])

model = load_prediction_model(MODEL_PATH)

if uploaded_file and model:
    pil_img = Image.open(uploaded_file)
    st.image(pil_img, caption="Uploaded Leaf", use_column_width=True)

    predicted, confidence, probs = predict_from_pil(pil_img, model)

    st.write(f"**Prediction:** {predicted}")
    st.write(f"**Confidence:** {confidence:.2f}")

    if confidence >= CONFIDENCE_THRESHOLD:
        english_advice = ADVICE.get(predicted, {}).get(farmland_size, "No advice available.")
        target_code = LANG_CODE[language]
        advice_text = translate_text_cached(english_advice, target_code)

        st.subheader("Recommended Actions")
        st.info(advice_text)

        audio_path = text_to_speech_cached(advice_text, target_code)
        if audio_path:
            st.audio(audio_path)
    else:
        st.warning("Low confidence prediction. Image queued for human review.")
        note = st.text_area("Optional note for review")
        if st.button("Queue for review"):
            saved_path = queue_for_review(pil_img, predicted, confidence, notes=note)
            st.success(f"Saved for review: {saved_path}")

2025-09-27 15:48:10.058 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-27 15:48:10.061 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-27 15:48:10.387 
  command:

    streamlit run C:\Users\HP\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-09-27 15:48:10.388 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-27 15:48:10.390 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-27 15:48:10.391 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-27 15:48:10.393 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running